In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
from PIL import Image
import numpy as np
# import pandas as pd

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))


In [ ]:
data_dir = "/kaggle/input/drone-detection-dataset/BirdVsDroneVsAirplane"

In [2]:
from torchvision import datasets, transforms
import torch
from torch.utils.data import DataLoader, random_split
from PIL import Image

transform = transforms.Compose([transforms.Resize((224, 224)),
                    transforms.ToTensor(),
                    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])])

def safe_loader(path):
    try:
        with open(path, "rb") as f:
            img = Image.open(f)
            return img.convert("RGB")
    except:
        return None

class SafeImageFolder(datasets.ImageFolder):
    def __getitem__(self, index):
        for i in range(len(self.samples)):
            idx = (index + i) % len(self.samples)
            path, target = self.samples[idx]
            sample = safe_loader(path)
            if sample is not None:
                if self.transform is not None:
                    sample = self.transform(sample)
                return sample, target
        raise RuntimeError("No valid images found in the dataset!")


def load_data(data_dir):
    dataset = SafeImageFolder(data_dir, transform=transform)
    return dataset


def split_dataloader(data,train_split):
    train_size = int(train_split * len(data))
    test_size = len(data) - train_size
    train_data, val_data = random_split(data, [train_size, test_size])
    trainL = DataLoader(train_data, batch_size=64, shuffle=True)
    valL = DataLoader(val_data, batch_size=64, shuffle=False)

    return trainL,valL


In [ ]:
dataset = load_data(data_dir)

In [ ]:
dataset

Dataset SafeImageFolder
    Number of datapoints: 2392
    Root location: /kaggle/input/drone-detection-dataset/BirdVsDroneVsAirplane
    StandardTransform
Transform: Compose(
               Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
               Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
           )

In [ ]:
train, val = split_dataloader(dataset, 0.8)

In [ ]:
for batch_idx, (features, labels) in enumerate(train):
    print(f"Batch {batch_idx+1} - Features shape: {features.shape}, Labels shape: {labels.shape}")
    break  # Remove this break to print all batches

Batch 1 - Features shape: torch.Size([64, 3, 224, 224]), Labels shape: torch.Size([64])


In [3]:
def build_model(num_classes=3, device="cuda"):
    # ResNet50 model and replace final layer
    model = models.resnet50(pretrained=True)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)

    model = model.to(device)
    return model

In [ ]:
def freeze_backbone(model):
    for param in model.parameters():
        param.requires_grad = False

    for param in model.fc.parameters():
        param.requires_grad = True

    return model

In [ ]:
def unfreeze_last_block(model):
    for param in model.layer4.parameters():
        param.requires_grad = True

    return model

In [ ]:
def get_optimizer(model, lr):
    return torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr
    )

In [ ]:
def get_loss_function():
    return nn.CrossEntropyLoss()

In [ ]:
def train_Func(model, dataloader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels)

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = correct.double() / len(dataloader.dataset)

    return epoch_loss, epoch_acc.item()

In [ ]:
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels)

    val_loss = running_loss / len(dataloader)
    val_acc = correct.double() / len(dataloader.dataset)

    return val_loss, val_acc.item()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [ ]:
num_classes = len(dataset.classes)
print("Classes:", dataset.classes)

Classes: ['Aeroplanes', 'Birds', 'Drones']


In [ ]:
model = build_model(num_classes, device)
model = freeze_backbone(model)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
optimizer = get_optimizer(model, lr=0.001)
criterion = get_loss_function()

In [ ]:
epochs = 20

for epoch in range(epochs):
    train_loss, train_acc = train_Func(model, train, optimizer, criterion, device)
    val_loss, val_acc = validate(model, val, criterion, device)


    print(f"Epoch {epoch+1}")
    print("Train Loss:", train_loss)
    print("Train Accuracy:", train_acc)
    print ("Val Loss:", val_loss)
    print("Val Accuracy:", val_acc)

Epoch 1
Train Loss: 0.2691843792796135
Train Accuracy: 0.9153162571876633
Val Loss: 0.21578697301447392
Val Accuracy: 0.9478079331941544
Epoch 2
Train Loss: 0.20362916787465413
Train Accuracy: 0.9435441714584422
Val Loss: 0.19251617975533009
Val Accuracy: 0.9540709812108559
Epoch 3
Train Loss: 0.19350693027178448
Train Accuracy: 0.9451123889179299
Val Loss: 0.18413317948579788
Val Accuracy: 0.9540709812108559
Epoch 4
Train Loss: 0.17892306173841158
Train Accuracy: 0.9456351280710925
Val Loss: 0.1781802736222744
Val Accuracy: 0.9478079331941544
Epoch 5
Train Loss: 0.16137910162409147
Train Accuracy: 0.9571353894406691
Val Loss: 0.1672139074653387
Val Accuracy: 0.9540709812108559
Epoch 6
Train Loss: 0.16213723570108413
Train Accuracy: 0.9513852587558808
Val Loss: 0.15583250112831593
Val Accuracy: 0.9686847599164926
Epoch 7
Train Loss: 0.15387292355298995
Train Accuracy: 0.9498170412963931
Val Loss: 0.15022540371865034
Val Accuracy: 0.9665970772442588
Epoch 8
Train Loss: 0.135993283738692

In [ ]:
torch.save(model.state_dict(), "resnet50_model.pth")
print("Model Saved Successfully ✅")

Model Saved Successfully ✅


In [4]:
import torch
import numpy as np

In [4]:

import torchvision.models as models
import torch.nn as nn
state_dict = torch.load("resnet50_model.pth", map_location="cpu")
model = build_model(num_classes=3, device="cpu")
model.load_state_dict(state_dict)
model.eval()

d:\Computer Vision NTI\session10\Drone_Classification\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\Computer Vision NTI\session10\Drone_Classification\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.load_state_dict(torch.load("resnet50_model.pth", map_location=device))
model.to(device)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [6]:
from data_preprocessing import preprocess_image
import numpy as np
import PIL.Image as Image
import cv2
img = cv2.imread("image.jpg")
img = preprocess_image(img)
print("Preprocessed Image Shape:", img.shape)
print("Preprocessed Image dtype:", img.dim())
print("Preprocessed Image size:", img.size(0))
# Ensure model input is [N, C, H, W]
if img.dim() == 5 and img.size(0) == 1:
    img = img.squeeze(0)
if img.dim() == 3:
    img = img.unsqueeze(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
img = img.to(device)
with torch.no_grad():
    output = model(img)

Preprocessed Image Shape: torch.Size([1, 3, 224, 224])
Preprocessed Image dtype: 4
Preprocessed Image size: 1


In [7]:
output = output.cpu().numpy()
predicted_class = np.argmax(output, axis=1)
print("Predicted Class:", predicted_class)

Predicted Class: [0]
